In [ ]:
!git clone https://github.com/ggerganov/llama.cpp

In [ ]:
model_id = "Qwen/Qwen1.5-1.8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id,
                                             dtype= torch.bfloat16,
                                             device_map="cuda")
draft_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen1.5-0.5B",
                                                   dtype=torch.bfloat16,
                                                   device_map="cuda")

In [ ]:
model.generation_config.cache_implementation='static'

In [ ]:
compiled_model = torch.compile(model,mode="reduce-overhead", fullgraph=True)

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
inputs = tokenizer("What is 2+2?", return_tensors='pt').to(device)

In [ ]:
inputs

In [ ]:
outputs = model.generate(**inputs , do_sample=True, temperature=0.7,
                         assistant_model=draft_model, max_new_tokens=64)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))